# 👗 Fit-Aware Virtual Try-On — Google Colab

This notebook runs the complete **Fit-Aware Virtual Try-On** pipeline on a Colab T4 GPU, with a public Streamlit URL via **ngrok** or **localtunnel**.

### Supported image-generation models
| Model | Key needed | Cost/image |
|---|---|---|
| Gemini 2.5 Flash Image (Nano Banana) | `GEMINI_API_KEY` | ~$0.039 |
| Gemini 3 Pro Image (Nano Banana Pro) | `GEMINI_API_KEY` | ~$0.134 |
| FLUX.1 Kontext Pro (fal.ai) | `FAL_KEY` | ~$0.04 |

### Runtime recommendation
- **Runtime → Change runtime type → T4 GPU** (needed for MediaPipe to run at acceptable speed)
- Free Colab gives ~2–3 h of T4 time per session

---
**Run all cells top-to-bottom.  Cell 7 prints the public URL.**

## Cell 1 — Check GPU

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU for best performance.")
else:
    print(gpu_info.stdout)
    print("✅ GPU is available.")

## Cell 2 — Clone or upload your project

**Option A (recommended):** push your project to a GitHub repo and clone it here.
**Option B:** use the Colab file manager (left panel) to upload a `.zip` and unzip it.

In [ ]:
import os

# ── OPTION A: clone from GitHub ──────────────────────────────────────────────
# Replace with your actual repo URL.
GITHUB_REPO = "https://github.com/YOUR_USERNAME/fit-aware-tryon.git"  # ← edit
PROJECT_NAME = "fit-aware-tryon"   # folder name after clone

USE_GITHUB = False   # ← set True to use GitHub clone

if USE_GITHUB:
    if os.path.isdir(PROJECT_NAME):
        print(f"Folder '{PROJECT_NAME}' already exists — pulling latest changes.")
        os.chdir(PROJECT_NAME)
        os.system("git pull")
    else:
        os.system(f"git clone {GITHUB_REPO} {PROJECT_NAME}")
        os.chdir(PROJECT_NAME)
    print("✅ Repo ready:", os.getcwd())

# ── OPTION B: zip upload ──────────────────────────────────────────────────────
else:
    print("Upload your project zip via the left file panel, then set the path below.")
    ZIP_PATH = "/content/fit-aware-tryon.zip"   # ← edit to match your upload
    EXTRACT_TO = "/content"

    if os.path.exists(ZIP_PATH):
        os.system(f"unzip -q -o '{ZIP_PATH}' -d '{EXTRACT_TO}'")
        # Find the extracted folder (first directory inside EXTRACT_TO that is not a system dir)
        dirs = [d for d in os.listdir(EXTRACT_TO)
                if os.path.isdir(os.path.join(EXTRACT_TO, d))
                and d not in ("sample_data",)]
        if dirs:
            project_dir = os.path.join(EXTRACT_TO, dirs[0])
            os.chdir(project_dir)
            print("✅ Extracted and changed to:", os.getcwd())
        else:
            print("Could not auto-detect project folder. Set os.chdir() manually below.")
    else:
        print(f"Zip not found at {ZIP_PATH}. Upload it first or switch to Option A.")

# After either option, verify the project root
print("\nDirectory listing:")
print(os.listdir("."))

## Cell 3 — Install dependencies

In [ ]:
%%bash
set -e

echo "=== Installing system packages ==="
apt-get install -q -y libgl1 libglib2.0-0 > /dev/null 2>&1

echo "=== Installing Python packages ==="
pip install -q \
    streamlit \
    mediapipe \
    google-genai \
    groq \
    pillow \
    numpy \
    opencv-python-headless \
    python-dotenv \
    fal-client \
    pyngrok

# Install project requirements if present
for req in requirements.txt api/requirements.txt web/requirements.txt; do
    if [ -f "$req" ]; then
        echo "Installing from $req"
        pip install -q -r "$req" 2>/dev/null || true
    fi
done

echo "=== All packages installed ==="
python -c "import streamlit, mediapipe, google.genai, groq, fal_client; print('Import check: OK')"

## Cell 4 — Download MediaPipe pose model

The model is ~7 MB and cached in `~/.cache/mediapipe/` so subsequent runs skip the download.

In [ ]:
from pathlib import Path
import urllib.request, os

MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)
MODEL_DIR  = Path.home() / ".cache" / "mediapipe"
MODEL_FILE = MODEL_DIR / "pose_landmarker_lite.task"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_FILE.exists() and MODEL_FILE.stat().st_size > 0:
    print(f"✅ Model already cached ({MODEL_FILE.stat().st_size / 1024:.0f} KB) — skipping download.")
else:
    print("Downloading MediaPipe pose model (~7 MB)…")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
    print(f"✅ Downloaded: {MODEL_FILE.stat().st_size / 1024:.0f} KB")

## Cell 5 — Configure API keys

Paste your keys in the fields below.  
They are written to a `.env` file in the project root — **never committed to git**.

| Key | Where to get it |
|---|---|
| `GEMINI_API_KEY` | https://aistudio.google.com/app/apikey |
| `GROQ_API_KEY` | https://console.groq.com/keys |
| `FAL_KEY` | https://fal.ai/dashboard/keys (only needed for FLUX.1 Kontext Pro) |

In [ ]:
import os
from pathlib import Path

# ─── Paste your keys here ─────────────────────────────────────────────────────
GEMINI_API_KEY   = ""   # Required for Gemini 2.5 Flash or Gemini 3 Pro
GROQ_API_KEY     = ""   # Required for LLM prompt writing
FAL_KEY          = ""   # Required only for FLUX.1 Kontext Pro
DJANGO_SECRET_KEY = "colab-dev-secret-key-not-for-prod"
# ─────────────────────────────────────────────────────────────────────────────

env_path = Path(".") / ".env"

env_lines = []
if GEMINI_API_KEY:
    env_lines.append(f"GEMINI_API_KEY={GEMINI_API_KEY}")
    env_lines.append(f"api_key={GEMINI_API_KEY}")   # legacy alias
if GROQ_API_KEY:
    env_lines.append(f"GROQ_API_KEY={GROQ_API_KEY}")
if FAL_KEY:
    env_lines.append(f"FAL_KEY={FAL_KEY}")
env_lines.append(f"DJANGO_SECRET_KEY={DJANGO_SECRET_KEY}")
env_lines.append("DJANGO_DEBUG=1")

env_path.write_text("\n".join(env_lines) + "\n")

# Also export to process env so the Streamlit subprocess inherits them
for line in env_lines:
    k, _, v = line.partition("=")
    os.environ[k] = v

print("✅ .env written with keys:")
for line in env_lines:
    key = line.split("=")[0]
    val = line.split("=", 1)[1]
    masked = val[:4] + "*" * max(0, len(val) - 4) if len(val) > 4 else "****"
    print(f"   {key} = {masked}")

## Cell 6 — (Optional) Upload garment images

If your `cloth/` folder is not part of the uploaded project, you can upload images here.  
Skip this cell if your project zip already contains the `cloth/` folder.

In [ ]:
from pathlib import Path

cloth_dir = Path("cloth")
cloth_dir.mkdir(exist_ok=True)
(Path("output")).mkdir(exist_ok=True)

existing = list(cloth_dir.glob("*.jpg")) + list(cloth_dir.glob("*.png"))
print(f"cloth/ currently has {len(existing)} image(s).")

catalog = Path("cloth/garments.json")
if not catalog.exists():
    print("⚠️  cloth/garments.json not found. The app needs this catalog file to show the wardrobe.")
    print("   Upload it via the file panel on the left, or include it in your project zip.")
else:
    import json
    cat = json.loads(catalog.read_text())
    print(f"✅ Catalog loaded: {len(cat.get('garments', []))} garment(s).")

## Cell 7 — Launch Streamlit + expose via ngrok

After running this cell, a **public URL** will appear below (e.g. `https://xxxx.ngrok-free.app`).  
Open it in any browser — no Colab window needed.

**Free ngrok** gives one simultaneous tunnel per account.  
Get a free auth-token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess, threading, time, os
from pathlib import Path
from pyngrok import ngrok, conf as ngrok_conf

# ── ngrok auth token (optional but recommended for stable tunnels) ─────────
NGROK_AUTH_TOKEN = ""   # ← paste your ngrok token, or leave empty for a short-lived tunnel
# ──────────────────────────────────────────────────────────────────────────

STREAMLIT_PORT = 8501
APP_FILE       = "app.py"   # path relative to CWD

if not Path(APP_FILE).exists():
    raise FileNotFoundError(
        f"{APP_FILE} not found in {os.getcwd()}. "
        "Make sure Cell 2 ran successfully and set the right project directory."
    )

# Configure ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any leftover Streamlit processes from a previous run
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

# Build the environment for the subprocess (inherits our keys)
env = os.environ.copy()
env["GLOG_minloglevel"] = "3"   # silence MediaPipe telemetry

# Start Streamlit in a background thread
def run_streamlit():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", APP_FILE,
            "--server.port",        str(STREAMLIT_PORT),
            "--server.headless",    "true",
            "--server.enableCORS",  "false",
            "--server.enableXsrfProtection", "false",
            "--browser.gatherUsageStats", "false",
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print("[streamlit]", line, end="")

import sys
t = threading.Thread(target=run_streamlit, daemon=True)
t.start()

# Wait for Streamlit to be ready
print("⏳ Waiting for Streamlit to start…")
time.sleep(8)

# Open ngrok tunnel
tunnel = ngrok.connect(STREAMLIT_PORT, "http")
public_url = tunnel.public_url

print("\n" + "="*60)
print(f"  ✅  App is live at: {public_url}")
print("="*60)
print("Open the URL above in any browser.")
print("This cell must stay running — do not interrupt it.")

## Cell 8 — (Alternative) Expose via localtunnel

If ngrok is down or you prefer not to create an account, use **localtunnel**.  
Skip this cell if Cell 7 worked.

In [ ]:
import subprocess, sys, os, time

STREAMLIT_PORT = 8501

# Install localtunnel (Node.js tool)
os.system("npm install -g localtunnel > /dev/null 2>&1")

# Kill leftover Streamlit if any
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

env = os.environ.copy()
env["GLOG_minloglevel"] = "3"

# Start Streamlit
subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port",     str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--browser.gatherUsageStats", "false",
    ],
    env=env,
)
time.sleep(8)

# Start localtunnel and print URL
lt = subprocess.Popen(
    ["lt", "--port", str(STREAMLIT_PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in lt.stdout:
    print(line, end="")
    if "https://" in line:
        url = line.strip().split()[-1]
        print(f"\n✅ App is live at: {url}")
        break

## Cell 9 — Quick smoke test (no UI)

Run this cell to verify the entire pipeline works end-to-end **before** opening the Streamlit app.  
It creates a dummy JPEG, runs the measurement pipeline on it, and calls the selected model.

Set `SMOKE_MODEL` to whichever model you want to test.

In [ ]:
import sys, os
from pathlib import Path

# Which model to smoke-test
# Options: "gemini-2.5-flash" | "gemini-3-pro" | "flux-kontext-pro"
SMOKE_MODEL = "gemini-2.5-flash"

sys.path.insert(0, str(Path(".").resolve()))

# ── 1. Synthetic person image (white 600×800 JPEG) ──────────────────────────
import numpy as np
from PIL import Image
from io import BytesIO

dummy_arr = np.ones((800, 600, 3), dtype=np.uint8) * 220
buf = BytesIO()
Image.fromarray(dummy_arr).save(buf, format="JPEG")
person_bytes = buf.getvalue()
print(f"Synthetic person image: {len(person_bytes)} bytes")

# ── 2. Fit calculation ───────────────────────────────────────────────────────
from src.fit_calculator import calculate_fit

person_dims = {"length": 66, "chest": 91, "shoulder": 38, "arm_length": 23}
fit = calculate_fit(person_dims, "G2")
print(f"Fit deltas: {fit['deltas']}")

# ── 3. Prompt building ───────────────────────────────────────────────────────
from src.prompt_builder import build_fit_prompt

prompt = build_fit_prompt(fit)
print(f"Prompt ({len(prompt.split())} words): {prompt[:120]}…")

# ── 4. Image generation ──────────────────────────────────────────────────────
garment_path = str(Path("cloth") / fit["image_path"])
output_path  = str(Path("output") / f"smoke_test_{SMOKE_MODEL}.png")

print(f"\nRunning generation with model: {SMOKE_MODEL}")

if SMOKE_MODEL in ("gemini-2.5-flash", "gemini-3-pro"):
    from src.gemini_client import generate_tryon
    import src.gemini_client as _gc
    model_name_map = {
        "gemini-2.5-flash": "gemini-2.5-flash-image",
        "gemini-3-pro":     "gemini-3.0-pro-image",
    }
    _gc.MODEL_NAME = model_name_map[SMOKE_MODEL]
    result = generate_tryon(
        person_image=person_bytes,
        garment_image_path=garment_path,
        prompt=prompt,
        output_path=output_path,
    )

elif SMOKE_MODEL == "flux-kontext-pro":
    import base64, fal_client, urllib.request
    p_b64 = base64.b64encode(person_bytes).decode()
    g_b64 = base64.b64encode(Path(garment_path).read_bytes()).decode()
    result_data = fal_client.subscribe(
        "fal-ai/flux-pro/kontext",
        arguments={
            "prompt":      prompt + " Dress the person with the reference garment.",
            "image_url":   f"data:image/jpeg;base64,{p_b64}",
            "image_url_2": f"data:image/jpeg;base64,{g_b64}",
            "num_inference_steps": 28,
            "guidance_scale": 3.5,
        },
    )
    img_url = result_data["images"][0]["url"]
    urllib.request.urlretrieve(img_url, output_path)
    result = output_path

print(f"\n✅ Smoke test passed! Image saved to: {result}")

# Display inline
from IPython.display import display
from PIL import Image as PILImage
display(PILImage.open(result).resize((400, 533)))

## Cell 10 — Stop the app

In [ ]:
import subprocess
from pyngrok import ngrok

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
print("✅ Streamlit and ngrok stopped.")

---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: mediapipe` | Re-run Cell 3 |
| `RuntimeError: No body detected` | The synthetic image in the smoke test is blank — this is expected. Use a real person photo in the actual app. |
| `No candidates` from Gemini | Usually a safety filter on the prompt/images — try a different garment or re-run with a shorter prompt (toggle off Groq in sidebar). |
| `FAL_KEY not found` | Fill in `FAL_KEY` in Cell 5, then re-run Cell 5. |
| ngrok `ERR_NGROK_8012` | You have another ngrok tunnel open. Run Cell 10 first, then re-run Cell 7. |
| `garments.json not found` | Upload it via the Colab file panel or include it in your project zip. |
| Slow MediaPipe on CPU | Make sure you selected **T4 GPU** runtime (Runtime → Change runtime type). |

---

## .env reference

```ini
GEMINI_API_KEY=your_gemini_key_here
api_key=your_gemini_key_here          # legacy alias — keep both
GROQ_API_KEY=your_groq_key_here
FAL_KEY=your_fal_key_here             # only needed for FLUX.1 Kontext Pro
DJANGO_SECRET_KEY=dev-only-secret
DJANGO_DEBUG=1
```